In 3–5 sentences: what did you build, what representation did you choose for the images, and what is the single most important thing you learned going from the linear baseline to the MLP?

I trained a linear softmax classifier and a two-hidden-layer ReLU MLP to recognize the 11 numeral characters. Both models turn each 64×64 grayscale image into 16×16 features by averaging 4×4 pixel blocks, then invert and scale the values to roughly [−1, 1]. The MLP has hidden widths 128 and 64 and produces 11 logits. The main lesson was that nonlinear hidden layers can combine stroke features that a single linear decision rule struggles to separate.


How many raw captures per character did you collect, and how did you keep the classes balanced? Any capture you redid and why?

I traced 442 original captures from the two guide fonts: 40 per character, except four and nine with 41 each. Every character × font cell has 20 captures, except fontA for four and nine with 21, so the classes are nearly balanced and every cell exceeds the required three. The Copybook export rendered each capture at three stroke widths (−3%, original, +3%), producing 1,326 exported images; adding 1,000 augmentations per exported image produced 1,327,326 images. I did not redo any captures.

![Coverage panel reconstructed from the Copybook export](copybook_coverage_reconstructed.png)


List both submission IDs (linear and MLP) and their scores.

| Submission | Server report | Score |
| --- | ---: | ---: |
| `jb-linear / my-linear-v3` | #10 | 0.710 |
| `jb-mlp / my-mlp-v5` | #23 | 0.795 |

Linear: my-linear-v3, #10, 0.710; MLP: my-mlp-v5, 23, 0.795

Analysis (graded): which font slice or confusion pair is weakest, and why do you think that happens? What would you try next?

The weakest hidden-set slice for the final MLP is fontA at 84.9%; fontB and fontD are 100.0%, and fontC is 99.6%. Its largest confusions are eight→six (46 of 200) and four→six (34 of 200). A plausible explanation is that some fontA stroke shapes make these characters look similar after 4×4 pooling, but the report does not include the individual images needed to confirm that. I would inspect those misclassified fontA images first, then test a change aimed at preserving the distinguishing strokes.


Compare linear vs MLP on the hidden set (accuracy, parameters, latency). Was the MLP worth its extra parameters everywhere, or only on some slices?

| Model and server report | Hidden accuracy | Float parameters | Median latency | Score |
| --- | ---: | ---: | ---: | ---: |
| Linear baseline, `my-linear-v3` (report #10) | 81.8% | 2,829 | 0.009 ms | 0.710 |
| Final MLP, `my-mlp-v5` (report #23) | 96.1% | 41,869 | 0.012 ms | 0.795 |

The MLP is 14.3 percentage points more accurate in these reports, using about 14.8× as many parameters and adding 0.003 ms to median latency. It is better on every reported slice: fontA rises from 70.2% to 84.9%, and rotated images from 82.8% to 96.9%. Both the nonlinear architecture and stronger augmentation settings may contribute to the difference, so these submissions do not isolate the effect of the architecture alone.


What is the single change you made that improved hidden accuracy the most — and what did you try that did NOT help?

The single change I credit most was increasing the augmentation ranges while keeping the original captures the same. The final generator permits rotations up to ±20°, translations up to ±15 pixels, scale changes up to ±15%, shear up to 0.05, and elastic displacement up to 5 pixels, while rejecting nearly blank results. The MLP hidden score rose from 83.2% for `my-mlp-v3` to 96.1% for `my-mlp-v5`; both used 4×4 pooling. A 2×2-pooled MLP did not help in the earlier `my-mlp-v2` submission (76.3% hidden), though that run was not a controlled pooling-only comparison.


Optional (worth up to 5 bonus points) — Is your dataset linearly separable?

With 11 classes in your chosen feature space (16×16 = 256 dims, or 8×8 = 64 dims, depending on your pooling factor), can a set of 11 linear decision boundaries (one per class, as softmax computes) perfectly partition all classes? What does the hidden-set result of your linear model tell you? A useful lens: Linear Discriminant Analysis (LDA) projects data onto directions that maximize between-class separation — if classes overlap in that projection, no linear boundary will separate them perfectly.

The 16×16 pooled representation has 256 features, and the linear-softmax model assigns one affine score to each of the 11 classes. Its 81.8% hidden accuracy shows that the fitted linear rule does not separate all hidden examples; it does **not** prove that no linear rule could separate the training captures. To establish separability of the finite training set, I would need to show zero training error with a linear classifier, or establish an infeasibility result for all linear classifiers. Errors on unseen fonts and rotations can also reflect a shift from the training distribution. LDA could visualize overlap in a discriminative projection, but overlap in that projection alone would not prove that the full 256-dimensional data are inseparable.
